# 04 — Evaluation & Results
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** take the final saved BERTopic model and its per-document topic
assignments (from notebook 03) and produce the actual results — topic overview, keyword
visualizations, how topics trend over time, and a sanity check against the original news
categories. This notebook's outputs map directly onto the report's "Results" section.

**Run this in Google Colab.** No GPU needed — everything here loads already-computed results
(the saved model, the topic-assignment CSV); nothing is refit or re-embedded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/topic-modelling-capstone'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
OUTPUTS_MODELS = f'{BASE_DIR}/outputs/models'
OUTPUTS_FIGURES = f'{BASE_DIR}/outputs/figures'


In [ ]:
!pip install -q bertopic pandas matplotlib seaborn wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42


## 1. Load the final model and topic assignments

Both were saved at the end of notebook 03 — the model itself (fitted on the full 284,916-document
dataset) and a copy of the preprocessed data with each headline's assigned `topic` column.


In [ ]:
best_model = BERTopic.load(f'{OUTPUTS_MODELS}/best_bertopic_model')
df = pd.read_csv(f'{DATA_PROCESSED}/headlines_with_topics.csv')

print(f"Loaded model and {len(df):,} topic-tagged documents.")
print(f"Number of topics (excluding outliers): {df['topic'].nunique() - (1 if -1 in df['topic'].values else 0)}")


## 2. Topic overview: sizes and keywords

`get_topic_info()` gives every topic's size and its top keywords in one table — this is the
master reference table for the report's results section.


In [ ]:
topic_info = best_model.get_topic_info()
topic_info.head(25)


In [ ]:
# Bar chart of the top 20 topics by size (excluding -1/outliers, shown separately since its
# scale would otherwise dwarf every real topic on the same chart).
top_n = 20
plot_df = topic_info[topic_info['Topic'] != -1].head(top_n).copy()
plot_df['label'] = plot_df['Topic'].astype(str) + ': ' + plot_df['Name'].str.split('_').str[1:4].str.join(' ')

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_df['label'], plot_df['Count'], color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Number of headlines')
ax.set_title(f'Top {top_n} topics by size (outliers excluded)')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topic_sizes_top20.png', dpi=150)
plt.show()

n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].values[0]
print(f"For reference — outlier count: {n_outliers:,} ({n_outliers/len(df):.1%} of all documents)")


## 3. Keyword breakdown for the largest topics

Word clouds per topic — a quick, intuitive way to present what each topic is actually about in
the report, beyond just a ranked keyword list.


In [ ]:
from wordcloud import WordCloud

top_topics = topic_info[topic_info['Topic'] != -1]['Topic'].head(9).tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, topic_id in zip(axes.flat, top_topics):
    words = dict(best_model.get_topic(topic_id))
    wc = WordCloud(width=400, height=300, background_color='white').generate_from_frequencies(words)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    topic_name = ' '.join(topic_info[topic_info['Topic'] == topic_id]['Name'].values[0].split('_')[1:4])
    ax.set_title(f"Topic {topic_id}: {topic_name}", fontsize=11)

plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topic_wordclouds_top9.png', dpi=150)
plt.show()


## 4. Topics over time

Since headlines span 2001–2023, tracking how topic volume shifts over the years gives real
historical insight — e.g. does a "COVID" topic spike exactly where expected? Does an election
-related topic show periodic peaks matching India's actual election calendar?


In [ ]:
df['publish_date'] = pd.to_datetime(df['publish_date'])
df['year'] = df['publish_date'].dt.year

# Focus on the 8 largest real topics for readability — more than that becomes unreadable on one chart.
top8_topics = topic_info[topic_info['Topic'] != -1]['Topic'].head(8).tolist()
topic_names = {
    t: ' '.join(topic_info[topic_info['Topic'] == t]['Name'].values[0].split('_')[1:3])
    for t in top8_topics
}

yearly_topic_counts = (
    df[df['topic'].isin(top8_topics)]
    .groupby(['year', 'topic'])
    .size()
    .unstack(fill_value=0)
    .rename(columns=topic_names)
)

fig, ax = plt.subplots(figsize=(13, 6))
yearly_topic_counts.plot(ax=ax, marker='o', markersize=3)
ax.set_title('Headline volume over time for the 8 largest topics')
ax.set_xlabel('Year')
ax.set_ylabel('Headline count')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topics_over_time.png', dpi=150)
plt.show()


## 5. Sanity check: discovered topics vs original news categories

The dataset came with its own `headline_category` labels (unused during modelling — BERTopic
never saw them). Checking whether each discovered topic's headlines mostly share one original
category is a good, independent sanity check that the model found *real* structure rather than
arbitrary clusters.


In [ ]:
# For each of the top topics, what's the dominant original category_top among its headlines?
category_check = []
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic'].head(15):
    subset = df[df['topic'] == topic_id]
    top_cat = subset['category_top'].value_counts()
    dominant_cat = top_cat.index[0]
    dominant_pct = top_cat.iloc[0] / len(subset)
    topic_name = ' '.join(topic_info[topic_info['Topic'] == topic_id]['Name'].values[0].split('_')[1:4])
    category_check.append({
        'topic_id': topic_id,
        'topic_keywords': topic_name,
        'size': len(subset),
        'dominant_original_category': dominant_cat,
        'pct_matching_dominant_category': round(dominant_pct, 3),
    })

category_check_df = pd.DataFrame(category_check)
category_check_df


## 6. What's in the outlier bucket?

~48% of headlines weren't assigned to any topic. Inspecting a random sample directly (rather
than just citing the percentage) helps distinguish "genuinely one-off, hard-to-cluster stories"
from any systematic pattern worth addressing.


In [ ]:
outlier_sample = df[df['topic'] == -1].sample(15, random_state=RANDOM_STATE)[['headline_text', 'category_top']]
outlier_sample


In [ ]:
# Category distribution within the outlier group, compared to the overall dataset —
# checks whether outliers skew toward specific categories (e.g. very long-tail local news)
# rather than being a random cross-section.
outlier_categories = df[df['topic'] == -1]['category_top'].value_counts(normalize=True).head(10)
overall_categories = df['category_top'].value_counts(normalize=True).head(10)

comparison = pd.DataFrame({
    'outlier_share': outlier_categories,
    'overall_share': overall_categories
}).fillna(0).round(3)
comparison


## 7. Save summary tables for the report

In [ ]:
topic_info.to_csv(f'{OUTPUTS_MODELS}/final_topic_info.csv', index=False)
category_check_df.to_csv(f'{OUTPUTS_MODELS}/topic_vs_category_check.csv', index=False)
print("Saved topic summary tables to Drive for use in the report.")


## 8. Summary of results

- **Final model**: 45 topics, 47.6% outliers (135,613 of 284,916 headlines), 2001-2023.
- **Largest topics**: politics/elections ("poll bjp congress", 17,081 headlines), cricket/sports
  ("cup world cup win", 14,958), education ("exam school university", 13,418), Bollywood
  ("film khan kapoor", 7,997), India-Pakistan/terrorism ("pak pakistan china", 6,620).
- **Topics-over-time — strong validation result**: the COVID topic is essentially flat/near-zero
  from 2001-2019, spikes sharply in 2020-2021, then recedes by 2022-23 — exactly matching the
  real pandemic timeline, discovered with zero date information given to the model. The
  elections topic ("poll bjp") shows distinct peaks in **2009, 2014, and 2019** — precisely
  India's Lok Sabha general election years. This is strong evidence the model captured genuine,
  real-world temporal structure rather than arbitrary clusters.
- **Category sanity check**: most topics align strongly with a single original
  `headline_category` (e.g. "stolen thief robbery" → 97.1% `city`-tagged, "suicide murder" →
  92.8%, "rape raped minor" → 88.8%). Some topics (elections, banking) show only moderate
  alignment with their dominant category (~58-62%) — but this reflects that the original `city`
  label is a broad catch-all covering ~60% of the entire dataset (per the EDA), not a modelling
  weakness. If anything, BERTopic surfaced more precise, semantically meaningful sub-structure
  (distinct political, financial, and legal topics) than the original coarse category labels
  provided on their own.
- **Outlier characterization**: the outlier group's category distribution closely mirrors the
  overall dataset (e.g. `city`: 63.4% of outliers vs 62.1% overall) — no systematic skew toward
  a particular news category. Manually inspecting outlier headlines (e.g. "Infant booked for
  stealing power", "32-yr-old Manipur supermom gives birth to 5 ba...") shows genuinely
  idiosyncratic, one-off local stories that don't recur often enough to form a stable topic —
  a legitimate characteristic of a diverse national news corpus, not a modelling failure.
  Notably, `sports` is *under*-represented among outliers (0.9% vs 3.5% overall), meaning sports
  headlines cluster unusually well — consistent with sports having consistent, repetitive
  vocabulary (team names, tournament terms).
- **Overall conclusion**: this is a usable, interpretable topic model for understanding two
  decades of Indian news coverage — the 45 discovered topics map cleanly onto recognizable,
  real-world news beats, and the topics-over-time analysis independently validates that the
  model captured real historical events (COVID, election cycles) without being told about them.
  The main limitation is the ~48% outlier rate; future work could explore hierarchical topic
  reduction (BERTopic's `reduce_outliers()`), running the full grid search on the complete
  dataset rather than a subsample (given more compute time/budget), or treating high-outlier
  categories like hyper-local `city` news as a deliberately separate, finer-grained modelling
  problem rather than expecting one global topic model to capture it.
